In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import sys

#math and array operations
import numpy as np
import math

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#data classes
import xarray as xr
import h5py
import pickle 

#loading bar
from tqdm import tqdm

#dates
from datetime import datetime, timedelta

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "MPAS_Model_Data", "VariableComparisons")
dataType = "SliceAverages"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
#Setup

# Region = "TRACER"; Case = "WET"; spinup_hours = "0"
# Region = "TRACER"; Case = "DIURNAL"; spinup_hours = "-5"
# Region = "PRECIP"; Case = "WET"; spinup_hours = "12"
# Region = "PRECIP"; Case = "DIURNAL"; spinup_hours = "12"

Region = "Hawaii"; Case = "TRADES"; spinup_hours = "12"

In [ ]:
#Load Model Directory Class
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

# RunType = (Region,Case,"TEMPO",spinup_hours)
# ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_DataSaving import DataSaving_Class

In [ ]:
#Importing PlottingModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_PlottingModelData import RadarPlotting_Class, LandMaskPlotting_Class

In [ ]:
LandMaskPlotting = LandMaskPlotting_Class(ModelData)
LandMaskPlotting.PlotLandMask_Test()

In [ ]:
#Importing Radar Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","Observation_Data"))
from CLASSES_RadarDataLoading import RadarData_MRMS_Class, RadarObservationMask_Class

sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_RadarDataPlotting import RadarPlotting_Class

In [ ]:
#IMPORT FUNCTIONS
# --- Add your Functions folder to sys.path ---
import sys
path = os.path.join(DirectoryManager.mainCodeDirectory, 'Functions_2.0')
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "AreaAverageFunctions",
    "ComputationFunctions",
    "DataFunctions",
    "DerivativeFunctions",
    "PlottingFunctions",
    "StatisticalFunctions",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [ ]:
####################################
#CALCULATION FUNCTIONS

In [ ]:
def InitiateMatrix(variableSubset, averageType, fill_nan=False):
    """
    Initializes an output matrix for a given variable subset.
    """
    if averageType == 'x':
        finalDimension = len(ModelData.latitude)
    elif averageType == 'y':
        finalDimension = len(ModelData.longitude)
    if "nVertLevels" in variableSubset.dims:
        shape = (ModelData.Ntime, ModelData.Nzc, finalDimension)

    elif "nVertLevelsP1" in variableSubset.dims:
        shape = (ModelData.Ntime, ModelData.Nzf, finalDimension)

    fill_value = np.nan if fill_nan else 0
    output = np.full(shape, fill_value, dtype=float)

    return output

def GetMean_x(variableSubset):
    variableMean = variableSubset.mean(dim=("longitude"), skipna=True).data
    return variableMean
def GetMean_y(variableSubset):
    variableMean = variableSubset.mean(dim=("latitude"), skipna=True).data
    return variableMean

In [ ]:
#Loading Radar Mask
RadarDataMask = RadarObservationMask_Class.LoadMaskData(DirectoryManager, ModelData)

In [ ]:
####################################
#CALCULATION FUNCTIONS

In [ ]:
def GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName):
    """
    Retrieves a variable subset from the given model data.
    If varName contains a '+', returns the sum of the two variables.
    """
    if '+' in varName:
        var1, var2 = varName.split('+')
        var1 = var1.strip()
        var2 = var2.strip()

        subset1 = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                      dataSubset_diag, dataSubset_static, var1)
        subset2 = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                      dataSubset_diag, dataSubset_static, var2)
        variableSubset = subset1 + subset2
    else:
        variableSubset = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                             dataSubset_diag, dataSubset_static, varName)

    return variableSubset

def MeanDBZ(variableSubset, GetMean):
    # Convert from dBZ → linear Z (mm^6 m^-3)
    variableSubset_power = 10 ** (variableSubset / 10.0)

    # Take mean in linear space
    variableMean = GetMean(variableSubset_power)

    # Convert mean Z → back to dBZ
    variableMean = 10.0 * np.log10(variableMean)

    return variableMean

In [ ]:
def RunCalculations(varNames, averageType):

    if averageType == 'x':
        GetMean = GetMean_x
    elif averageType == 'y':
        GetMean = GetMean_y
    
    outputDictionary={}
    
    num_times = ModelData.Ntime
    for count, t in enumerate(tqdm(range(num_times), desc="Processing timesteps")):
        # if t % 10 == 0: print(f"Currently working on time {t}/{num_times}","\n")
            
        #Loading Data
        [dataSubset, dataSubset_diag, dataSubset_static, lat, lon, _, _] = DataOperator_Class.GetData_Subset(ModelData, t)

        for varName in varNames:
            if count == 0: print(f"Running for {varName}")
            #Subsetting Data

            variableSubset= GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName)

            if varName in ['refl10cm']:
                variableSubset = variableSubset.where(variableSubset > 0)

            #Applying RadarDataMask
            variableSubset = variableSubset.where(RadarDataMask == True)
            
            #Initializing Output
            if count == 0:
                output = InitiateMatrix(variableSubset, averageType, fill_nan=False)
                outputDictionary[varName] = output

            #Taking Mean
            if varName in ['refl10cm']:
                variableMean = MeanDBZ(variableSubset, GetMean)  
            else:
                variableMean = GetMean(variableSubset)
                
            outputDictionary[varName][t] = variableMean

    return outputDictionary

# Notes:
# (1) may need to subset land/water later

In [ ]:
def RunAreaAverages(ModelData,varNames,averageType,name):
    filePath = DataOperator_Class.GetOutputFilePath(ModelData, DirectoryManager, outputDirectory, fileName = f"outputDictionary_{name}.h5")
    
    #loading back in 
    try:
        outputDictionary = DataSaving_Class.LoadDictionaryFromH5(filePath)
        return outputDictionary
    except Exception as e:
        print(f"Error: {e}")
        
        print("Running Calculation")
        outputDictionary = RunCalculations(varNames,averageType) #takes about 10 minutes
        #saving output
        
        DataSaving_Class.SaveDictionaryToH5(outputDictionary, filePath)
        return outputDictionary

In [ ]:
###############
#Loading in MRMS RadarTimeseries
###############

def LoadRadarTimeseries(ModelData):
    """
    Build the time-series filename using ModelData and load the .pkl file.
    Creates output directory if needed.
    """

    # Build file name
    fileName = (
        f"RadarTimeseries_{ModelData.region}_"
        f"{ModelData.case}_spinup{ModelData.spinup_hours}hrs.pkl"
    )

    # Build directory for radar timeseries
    outputDir = os.path.join(
        DirectoryManager.GetOutputDirectory(codeType='DataAnalysis/Observation_Data', dataType='RadarComparison'),
        "RadarTimeseries"
    )
    os.makedirs(outputDir, exist_ok=True)

    # Full path to the .pkl file
    fullFilePath = os.path.join(outputDir, fileName)

    # Try to load existing file
    if os.path.exists(fullFilePath):
        print(f"Loading existing file: {fullFilePath}")
        with open(fullFilePath, "rb") as f:
            return fullFilePath, pickle.load(f)

    # No file found
    return fullFilePath, None

def Add_MRMS_RadarTimeSeries_Plot(ax, loc='lower right'):
    ax.plot([datetime.strptime(t, "%Y-%m-%d_%H.%M.%S") for t in ModelData.timeStrings], MRMS_RadarTimeseries, color='black',label='MRMS')
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, frameon=True, fontsize=9, loc=loc)

fileName, list_array = LoadRadarTimeseries(ModelData)
if list_array is not None:
    MRMS_RadarTimeseries = list_array[:,2]

In [ ]:
####################################
#CALCULATING FUNCTIONS

In [ ]:
def GetVarNames():
    #3D Variables (9 vars)
    #microphysics variables
    varNames = ["refl10cm","qv", "qc+qi", "qr", "qg", "relhum"]
    #convection variables
    varNames += ["w", "theta", "divergence"]
    return varNames

#running
def GetDictionary_x(ModelData):
    varNames = GetVarNames()
    outputDictionary = RunAreaAverages(ModelData,varNames, "x", "1")
    return outputDictionary

#running
def GetDictionary_y(ModelData):
    varNames = GetVarNames()
    outputDictionary = RunAreaAverages(ModelData,varNames, "y", "2")
    return outputDictionary

In [ ]:
####################################
#PLOTTING FUNCTIONS

In [ ]:
def GetVerticalCoord(dataSubset):
    pressure_profile = dataSubset['pressure'].mean(dim=("latitude","longitude")).data
    dp = pressure_profile[-1] - pressure_profile[-2]
    p_topface = pressure_profile[-1] + dp  # extrapolate linearly
    pressure_profile_face = np.append(pressure_profile, p_topface)
    return (pressure_profile/100,pressure_profile_face/100)

[dataSubset, dataSubset_diag, dataSubset_static, lat, lon, _, _] = DataOperator_Class.GetData_Subset(ModelData, t=0)
pressure_profiles = GetVerticalCoord(dataSubset)
time_strings = ModelData.timeStrings
time = [datetime.strptime(t, "%Y-%m-%d_%H.%M.%S") for t in time_strings]

In [ ]:
#Helper Functions

def nansubtract(a, b):
    """
    Element-wise subtraction (a - b) that preserves NaNs.

    If shapes differ, raises a ValueError.
    """
    if a.shape != b.shape:
        raise ValueError(f"Shape mismatch: a{a.shape} != b{b.shape}")

    return np.where(np.isnan(a) | np.isnan(b), np.nan, a - b)

# Example: align datetime x-limits to min/max of your data
def SetXLimitsDatetime(ax, time_array):
    """
    Ensures datetime x-axis starts and ends exactly at the first and last time values.
    Works for both datetime.datetime and np.datetime64 arrays.
    """
    import numpy as np
    from matplotlib.dates import date2num

    # Convert to Matplotlib’s internal float format if needed
    times = np.asarray(time_array)
    if np.issubdtype(times.dtype, np.datetime64):
        times = date2num(times)
    elif isinstance(times[0], (object,)):
        try:
            times = date2num(times)
        except Exception:
            pass

    ax.set_xlim(times.min(), times.max())

def AlignAxesRight(ax_list):
    """
    Aligns the right edges of all axes in ax_list (e.g., contour + line plots),
    so that colorbars don't make some axes narrower.

    It uses the first axis that contains a contour or image
    (typically a contourf plot) as the reference width.
    """

    # Try to find a contour axis (has .collections or .images)
    ref_ax = None
    for ax in ax_list:
        if getattr(ax, "collections", []) or getattr(ax, "images", []):
            ref_ax = ax
            break

    # If no contour axis found, just use the first axis
    if ref_ax is None:
        ref_ax = ax_list[0]

    ref_pos = ref_ax.get_position()

    # Apply its width to all other axes
    for ax in ax_list:
        pos = ax.get_position()
        new_pos = [pos.x0, pos.y0, ref_pos.width, pos.height]
        ax.set_position(new_pos)

    print(f"Aligned {len(ax_list)} axes using reference width from contour axis at {ref_pos.width:.3f}")
# #EXAMPLE USAGE
# fig, axs = plt.subplots(2, 1, figsize=(8, 6))

# # contourf on top, line on bottom
# time = np.arange(24)
# pressure = np.linspace(1000, 100, 25)
# data = np.sin(time / 3)[None, :] * np.exp(-pressure[:, None] / 1000)

# plot = axs[0].contourf(time, pressure, data, cmap="RdBu_r")
# plt.colorbar(plot, ax=axs[0], orientation="vertical", pad=0.02)
# axs[1].plot(time, np.sin(time / 3), color="k")

# # Align both
# AlignAxesRight(axs)

# plt.show()

from matplotlib.ticker import MultipleLocator
def add_minor_white_grid(ax, alpha=0.5, lw=1.0, thickness=1.4, color='lightgray'):
    """
    Add white semi-transparent grid lines halfway between major ticks
    on both x and y axes (for contour plots).
    """
    from matplotlib.ticker import MultipleLocator

    # --- Minor locators at half the major spacing ---
    try:
        major_x = ax.xaxis.get_major_locator()
        step_x = major_x()[1] - major_x()[0]
        ax.xaxis.set_minor_locator(MultipleLocator(step_x / 2))
    except Exception:
        pass

    try:
        major_y = ax.yaxis.get_major_locator()
        step_y = major_y()[1] - major_y()[0]
        ax.yaxis.set_minor_locator(MultipleLocator(step_y / 2))
    except Exception:
        pass

    # --- Grid styling ---
    ax.grid(True, which="major", color=color, alpha=alpha, lw=lw * thickness)
    ax.grid(True, which="minor", color=color, alpha=alpha, lw=lw)

def AdjustLayout(fig,
                 left=0.07, right=0.97, bottom=0.07,
                 wspace=0.35, hspace=0.6,
                 title_space_inches=0.9, 
                 title_y_inches_from_top=0.25):
    """
    Applies a robust manual Matplotlib layout
    to a figure, reserving absolute space for a suptitle.
    """
    
    # Get figure height in inches
    fig_height_inches = fig.get_figheight()
    
    # Calculate the 'top' margin (where plots end) in relative figure coords
    # This leaves 'title_space_inches' at the top.
    top_margin = 1.0 - (title_space_inches / fig_height_inches)
    
    # Calculate the 'y' position for the suptitle
    title_y_relative = 1.0 - (title_y_inches_from_top / fig_height_inches)
    
    # Apply the manual layout
    plt.subplots_adjust(left=left, right=right, bottom=bottom, 
                        top=top_margin, wspace=wspace, hspace=hspace)

    # Return the calculated 'y' coordinate for the suptitle
    return title_y_relative

In [ ]:
def SaveFigure(fig, combinedDict, key,label_text):
    """
    Saves a figure to the appropriate directory based on the models in combinedDict.
    """
    # --- Define output subdirectory and file path ---
    outputSubDirectory = f"{ModelData.region}_{ModelData.case}_{label_text}_{ModelData.spinup_hours}hrs"
    os.makedirs(os.path.join(outputPlottingDirectory, outputSubDirectory), exist_ok=True)

    outputFile = os.path.join(
        outputPlottingDirectory,
        outputSubDirectory,
        f"CombinedPlot_{key}.png"
    )

    # --- Save figure ---
    fig.savefig(outputFile, dpi=100, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved to {outputFile}")

In [ ]:
def PlotSingle(axis, outputDictionarys, varName, labels,
               plottype="ZX", clim=None,num_levels=21):
    """
    Plot one variable on a given Matplotlib axis.
    Supports either:
      - A single dictionary (for single-model plots)
      - Two dictionaries (for model comparisons or line overlays)
    clim: tuple (vmin, vmax) for consistent color scaling (ignored for reflectivity)
    """

    # ------------------------------------------------------
    #  Helper: Line Plot
    # ------------------------------------------------------
    def lineplot(xAxis,xLabel, output, varName, units, color, label):
        axis.plot(xAxis, output.squeeze(), color=color, label=label)
        axis.set_ylabel(f"{varName} " + fr"$({units})$")
        axis.set_xlabel(xLabel)
        axis.grid(True)

    # ------------------------------------------------------
    #  Helper: Consistent Colorbar Formatting
    # ------------------------------------------------------
    def add_colorbar(fig, mappable, ax, label, ticks=None, orientation="vertical"):
        """Add a consistently styled, larger colorbar."""
        cbar = fig.colorbar(
            mappable, ax=ax, orientation=orientation,
            fraction=0.12, pad=0.020, aspect=20, shrink=1.15
        )
        cbar.set_label(label, fontsize=11)
        cbar.ax.tick_params(labelsize=8, width=1.1, length=4, pad=2)
        if ticks is not None:
            cbar.set_ticks(ticks)
        # Prevent overcrowding
        if len(cbar.get_ticks()) > 10:
            from matplotlib.ticker import MaxNLocator
            cbar.ax.yaxis.set_major_locator(MaxNLocator(8))
        return cbar

    # ------------------------------------------------------
    #  Units and scaling
    # ------------------------------------------------------
    units = ModelData.GetUnits_Specific(varName).replace(" ", r"\ ")
    axisTitle = varName.replace("divergence", "convergence")
    if varName in ["qv", "qc", "qi", "qr", "qg", "q2", "qfx"]:
        multiplier = 1e3
        units = units.replace('kg', 'g', 1)
    elif varName == "divergence":
        multiplier = -1
    else:
        multiplier = 1

    # ------------------------------------------------------
    #  Plotting Coordinates
    # ------------------------------------------------------
    # sample_dict = outputDictionarys[0]
    # output_sample = sample_dict[varName]
    # pressure_profile = (
    #     pressure_profiles[0]
    #     if output_sample.shape[1] == pressure_profiles[0].shape[0]
    #     else pressure_profiles[1]
    # )
    z_levels_filePath = "/glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.1/TRACER/WET/MPAS-Model_8.3.1_56nz/zeta_30km_57levels.txt"
    zlevels = np.loadtxt(z_levels_filePath)/1e3
    if varName not in ['w']:
        zlevels_plot = 0.5 * (zlevels[:-1] + zlevels[1:])
    else:
        zlevels_plot = zlevels.copy()

    xAxis = ModelData.latitude if 'Y' in plottype else ModelData.longitude
    xLabel = "Latitude" if 'Y' in plottype else "Longitude"
    mean_axis = 0
    
    # ------------------------------------------------------
    #  Case 1: Single-model plotting
    # ------------------------------------------------------
    if len(outputDictionarys) == 1:
        output = multiplier * outputDictionarys[0][varName]

        # Choose color setup
        if varName in ["w", "divergence"]:
            cmap = "RdBu_r"
        elif varName in ["refl10cm", "refl10cm_1km"]:
            cmap = None  # handled separately
        else:
            cmap = "viridis"

        # --- Line plot ---
        if output.ndim == 1 or output.shape[1] == 1:
            color = "k"
            label = labels[0] if labels else None
            lineplot(xAxis, xLabel, output, axisTitle, units, color, label)

        # --- Contour plot ---
        else:
            if plottype in ["ZX","ZY"] and varName not in ["refl10cm", "refl10cm_1km"]:
                # Apply shared clim via levels
                if clim is not None:
                    c0 = multiplier * clim[0]
                    c1 = multiplier * clim[1]
                    cmin, cmax = sorted([c0, c1])
                    levels = np.linspace(cmin, cmax, num_levels)
                    # levels = multiplier*np.linspace(clim[0], clim[1], num_levels)
                else:
                    levels = num_levels

                # Symmetric norm for diverging fields
                if varName in ["w", "divergence"]:
                    norm = TwoSlopeNorm(vcenter=0.0, vmin=clim[0] if clim else np.nanmin(output),
                                        vmax=clim[1] if clim else np.nanmax(output))
                else:
                    norm = None

                plot = axis.contourf(xAxis, zlevels_plot, output, cmap=cmap,
                                     levels=levels, norm=norm, extend="both")
                cbar = add_colorbar(axis.figure, plot, axis,
                             label=f"{axisTitle} " + fr"$({units})$")
                add_minor_white_grid(axis)
                axis.set_ylabel("Altitude (km)")
                axis.set_xlabel(xLabel)
                axis.set_ylim(0,20)
                # axis.invert_yaxis()

            elif plottype in ["ZX","ZY"] and varName in ["refl10cm", "refl10cm_1km"]:
                cmap, norm, levels, ticks = RadarPlotting_Class.GetReflectivityColormap()
                plot = axis.contourf(xAxis, zlevels_plot, output,
                                     levels=levels, cmap=cmap, norm=norm, extend='both')
                cbar = add_colorbar(axis.figure, plot, axis,
                                    label="Reflectivity (dBZ)", ticks=ticks)
                add_minor_white_grid(axis)
                RadarPlotting_Class.FormatReflectivityColorbar(
                    cbar, ticks, orientation='vertical', show_labels=False
                )
                axis.set_ylabel("Altitude (km)")
                axis.set_xlabel(xLabel)
                axis.set_ylim(0,20)
                # axis.invert_yaxis()

            elif plottype in ["X","Y"]:
                mean_output = np.nanmean(output, axis=mean_axis)
                color = "k"
                label = labels[0] if labels else None
                lineplot(xAxis, xLabel, mean_output, axisTitle, units, color, label)

    # ------------------------------------------------------
    #  Case 2: Two-model plotting
    # ------------------------------------------------------
    else:
        output1 = multiplier * outputDictionarys[0][varName]
        output2 = multiplier * outputDictionarys[1][varName]
        label1, label2 = labels

        is_line = (
            output1.ndim == 1 and output2.ndim == 1
            or output1.shape[1] == 1 and output2.shape[1] == 1
        )

        if is_line:
            with np.errstate(invalid="ignore"):
                mean1 = np.nanmean(output1, axis=mean_axis) if output1.ndim > 1 else output1
                mean2 = np.nanmean(output2, axis=mean_axis) if output2.ndim > 1 else output2
            lineplot(xAxis, xLabel, mean1, axisTitle, units, "blue", label1)
            lineplot(xAxis, xLabel, mean2, axisTitle, units, "green", label2)
            axis.legend(loc="upper left")

        else:
            if plottype in ["ZX","ZY"]:
                diff = nansubtract(output1, output2)
                cmap = plt.get_cmap("RdBu_r").copy()
                cmap.set_bad("black")
                axis.set_facecolor('black')
                vlim = np.nanmax(np.abs(diff))
                norm = TwoSlopeNorm(vcenter=0.0, vmin=-vlim, vmax=vlim)
                plot = axis.contourf(xAxis, zlevels_plot, diff, cmap=cmap,
                                     levels=np.linspace(-vlim, vlim, 40),
                                     norm=norm, extend="both")
                cbar = add_colorbar(axis.figure, plot, axis,
                             label=f"Δ{axisTitle} " + fr"$({units})$")
                axis.set_ylabel("Altitude (km)")
                axis.set_xlabel(xLabel)
                axis.set_ylim(0,20)
                # axis.invert_yaxis()

            elif plottype in ["X","Y"]:
                with np.errstate(invalid="ignore"):
                    mean1 = np.nanmean(output1, axis=mean_axis)
                    mean2 = np.nanmean(output2, axis=mean_axis)
                lineplot(xAxis, xLabel, mean1, axisTitle, units, "blue", label1)
                lineplot(xAxis, xLabel, mean2, axisTitle, units, "green", label2)
                axis.legend(loc="upper left")

    if varName in ["qr","qg","qc+qi","w"]:
        if plottype in ["Y","X"]:
            apply_scientific_notation([axis],dim='y')
        elif plottype in ["ZY","ZX"]:
            apply_scientific_notation_colorbar([cbar])
            
    # ------------------------------------------------------
    #  Title and finish
    # ------------------------------------------------------
    axis.set_title(axisTitle, fontsize=11)

In [ ]:
def MakeCombinedPlot(combinedDict, varNames, key, plottype):
    """
    combinedDict = {
        "X": {"NSSL": dict, "TEMPO": dict},
        "Y": {...},
        "ZX": {...},
        "ZY": {...}
    }

    For ZX/ZY:
        3-column layout: Model1 / Model2 / Difference

    For X/Y:
        overlay layout: both models on 1 axis
    """

    combinedDict2 = combinedDict[key]
    modelLabels = list(combinedDict2.keys())  # ["NSSL", "TEMPO"]
    first_model = modelLabels[0]
    # varNames = list(combinedDict2[first_model].keys())
    n_vars = len(varNames)

    # ----------------------------------------------------------
    # Layout logic
    # ----------------------------------------------------------
    if plottype in ["ZX", "ZY"]:
        n_cols = 3
        n_rows = n_vars
        layout_mode = "comparison"     # Model1, Model2, Δ
    else:
        n_cols = 3
        n_rows = int(np.ceil(n_vars / n_cols))
        layout_mode = "overlay"        # X/Y overlay

    fig = plt.figure(figsize=(5.5 * n_cols, 3.5 * n_rows))
    gs = gridspec.GridSpec(n_rows, n_cols, figure=fig,
                           wspace=0.3, hspace=0.6)

    # ----------------------------------------------------------
    # Loop variables
    # ----------------------------------------------------------
    for i, varName in enumerate(varNames):
        axisTitle = varName.replace("divergence", "convergence")

        # ======================================================
        # Comparison layout (ZX, ZY)
        # ======================================================
        if layout_mode == "comparison":
            row = i

            # Compute clim
            if varName not in ["refl10cm", "refl10cm_1km"]:
                out1 = combinedDict2[modelLabels[0]][varName]
                out2 = combinedDict2[modelLabels[1]][varName]
                vmin = np.nanmin([np.nanmin(out1), np.nanmin(out2)])
                vmax = np.nanmax([np.nanmax(out1), np.nanmax(out2)])
                clim = (vmin, vmax)
            else:
                clim = None

            # --- Model 1 ---
            ax1 = fig.add_subplot(gs[row, 0])
            PlotSingle(ax1, [combinedDict2[modelLabels[0]]],
                       varName,
                       labels=[modelLabels[0]],
                       plottype=plottype,
                       clim=clim)
            ax1.set_title(f"{modelLabels[0]} {axisTitle}")

            # --- Model 2 ---
            ax2 = fig.add_subplot(gs[row, 1])
            PlotSingle(ax2, [combinedDict2[modelLabels[1]]],
                       varName,
                       labels=[modelLabels[1]],
                       plottype=plottype,
                       clim=clim)
            ax2.set_title(f"{modelLabels[1]} {axisTitle}")

            # --- Difference ---
            ax3 = fig.add_subplot(gs[row, 2])
            PlotSingle(ax3, [
                combinedDict2[modelLabels[0]],
                combinedDict2[modelLabels[1]]
            ],
                       varName,
                       labels=modelLabels,
                       plottype=plottype)
            ax3.set_title(f"Δ({modelLabels[0]} - {modelLabels[1]}) {axisTitle}")

        # ======================================================
        # Overlay layout (X, Y)
        # ======================================================
        else:
            row, col = divmod(i, n_cols)
            ax = fig.add_subplot(gs[row, col])

            colors = {"NSSL": "blue", "TEMPO": "green"}

            for label in modelLabels:
                PlotSingle(ax, [combinedDict2[label]],
                           varName,
                           labels=[label],
                           plottype=plottype)
                ax.lines[-1].set_color(colors[label])
                ax.lines[-1].set_label(label)

            ax.legend()
            ax.set_title(axisTitle)

    # ----------------------------------------------------------
    # Formatting
    # ----------------------------------------------------------
    # for ax in fig.get_axes():
    #     plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

    # Tighten layout
    fig.subplots_adjust(top=0.95)

    # fig.suptitle(f"{ModelData.region}_{ModelData.case} "
    #              f"{modelLabels[0]} vs {modelLabels[1]}",
    #              fontsize=16, fontweight="bold")

    return fig, combinedDict2, key, f"{modelLabels[0]}_vs_{modelLabels[1]}"


In [ ]:
def SaveFigure(fig, combinedDict, key,label_text):
    """
    Saves a figure to the appropriate directory based on the models in combinedDict.
    """
    # --- Define output subdirectory and file path ---
    outputSubDirectory = f"{ModelData.region}_{ModelData.case}_{label_text}_{ModelData.spinup_hours}hrs"
    os.makedirs(os.path.join(outputPlottingDirectory, outputSubDirectory), exist_ok=True)

    outputFile = os.path.join(
        outputPlottingDirectory,
        outputSubDirectory,
        f"SliceAverages_{key}.png"
    )

    # --- Save figure ---
    fig.savefig(outputFile, dpi=100, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved to {outputFile}")

In [ ]:
####################################
#CALCULATING

In [ ]:
#getting NSSL dictionaries
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

outputDictionary_x_NSSL = GetDictionary_x(ModelData)
outputDictionary_y_NSSL = GetDictionary_y(ModelData)


#getting TEMPO dictionaries
RunType = (Region,Case,"TEMPO",spinup_hours)
ModelData = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

outputDictionary_x_TEMPO = GetDictionary_x(ModelData)
outputDictionary_y_TEMPO = GetDictionary_y(ModelData)

In [ ]:
#Removing Levels Above 20 km from Timeseries Averages
def SubsetAltitude(Dictionary):

    for varName, dataArray in Dictionary.items(): 
        
        #Getting Z Threshold Indexes
        z_levels_filePath = "/glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.1/TRACER/WET/MPAS-Model_8.3.1_56nz/zeta_30km_57levels.txt"
        zlevels = np.loadtxt(z_levels_filePath)/1e3
        zlevels_center = 0.5 * (zlevels[:-1] + zlevels[1:])
        zc_20 = np.where(zlevels_center>20)[0][0]
        zf_20 = np.where(zlevels>20)[0][0]
        z_20 = zf_20 if varName == "w" else zc_20
        
        #Applying Nan to Altitudes Greater than 20 km
        
        Dictionary[varName][:,z_20+1:,:] = np.nan

    return Dictionary


def FindIndexAtHourOffset(timeStrings, hourOffset):
    # Convert strings → datetime objects
    dt_list = [datetime.strptime(t, "%Y-%m-%d_%H.%M.%S") for t in timeStrings]
    
    # Starting time
    start = dt_list[0]
    target = start + timedelta(hours=hourOffset)
    
    # Find closest timestamp
    closest_dt = min(dt_list, key=lambda x: abs(x - target))
    idx = dt_list.index(closest_dt)
    
    return idx, timeStrings[idx]

def SubsetTime(Dictionary):
    t12, _ = FindIndexAtHourOffset(ModelData.timeStrings, 12)
    for varName, dataArray in Dictionary.items(): 
        Dictionary[varName][:t12] = np.nan

    return Dictionary

#Subsetting Altitude
outputDictionary_x_NSSL = SubsetAltitude(outputDictionary_x_NSSL)
outputDictionary_y_NSSL = SubsetAltitude(outputDictionary_y_NSSL)
outputDictionary_x_TEMPO = SubsetAltitude(outputDictionary_x_TEMPO)
outputDictionary_y_TEMPO = SubsetAltitude(outputDictionary_y_TEMPO)

#Subsetting Altitude
outputDictionary_x_NSSL = SubsetTime(outputDictionary_x_NSSL)
outputDictionary_y_NSSL = SubsetTime(outputDictionary_y_NSSL)
outputDictionary_x_TEMPO = SubsetTime(outputDictionary_x_TEMPO)
outputDictionary_y_TEMPO = SubsetTime(outputDictionary_y_TEMPO)

In [ ]:
#Taking Time Average
outputDictionarys = [
    outputDictionary_x_NSSL,
    outputDictionary_y_NSSL,
    outputDictionary_x_TEMPO,
    outputDictionary_y_TEMPO
]

for outputDictionary in outputDictionarys:
    for key in outputDictionary.keys():
        arr = outputDictionary[key]
        outputDictionary[key] = np.nanmean(arr, axis=0)   # collapse t dimension → (y, x)
        
        #t=60; outputDictionary[key] = arr[t]

In [ ]:
####################################
#PLOTTING

In [ ]:
combinedDictionary = {
    "X": {
        "NSSL": outputDictionary_x_NSSL,
        "TEMPO": outputDictionary_x_TEMPO
    },
    "Y": {
        "NSSL": outputDictionary_y_NSSL,
        "TEMPO": outputDictionary_y_TEMPO
    },
    "ZX": {
        "NSSL": outputDictionary_x_NSSL, 
        "TEMPO": outputDictionary_x_TEMPO
    },
    "ZY": {
        "NSSL": outputDictionary_y_NSSL,
        "TEMPO": outputDictionary_y_TEMPO
    }
}

varNames = ['qc+qi', 'qg', 'qr', 'qv', 'refl10cm', 'relhum', 'theta', 'w', 'divergence']

In [ ]:
[fig, combinedDict2, key, label_text] = MakeCombinedPlot(combinedDictionary, varNames, key = 'X', plottype="X")
LandMaskPlotting.ApplyPlotLandMask(fig,coordType='x',plotType="line")
SaveFigure(fig, combinedDict2, key, label_text)

In [ ]:
[fig, combinedDict2, key, label_text] = MakeCombinedPlot(combinedDictionary, varNames, key = 'Y', plottype="Y")
LandMaskPlotting.ApplyPlotLandMask(fig,coordType='y',plotType="line")
SaveFigure(fig, combinedDict2, key, label_text)

In [ ]:
#ZX Variable Plots
[fig, combinedDict2, key, label_text] = MakeCombinedPlot(combinedDictionary, varNames, key = 'ZX', plottype="ZX")
LandMaskPlotting.ApplyPlotLandMask(fig,coordType='x',plotType="contourf")
SaveFigure(fig, combinedDict2, key, label_text)

In [ ]:
#ZY Variable Plots
[fig, combinedDict2, key, label_text] = MakeCombinedPlot(combinedDictionary, varNames, key = 'ZY', plottype="ZY")
LandMaskPlotting.ApplyPlotLandMask(fig,coordType='y',plotType="contourf")
SaveFigure(fig, combinedDict2, key, label_text)